# **Assignment 4.1 - Model Store**

In [ ]:
%pip install "sagemaker<3.0.0"

In [ ]:
import os
import boto3
import sagemaker

role = sagemaker.get_execution_role()
sess = sagemaker.Session()
region = sess.boto_region_name

bucket = sess.default_bucket()
prefix = "DEMO-breast-cancer-prediction-xgboost-highlevel"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [ ]:
import pandas as pd
import numpy as np

s3 = boto3.client("s3")

filename = "wdbc.csv"
s3.download_file(
    f"sagemaker-example-files-prod-{region}", "datasets/tabular/breast_cancer/wdbc.csv", filename
)
data = pd.read_csv(filename, header=None)

# specify columns extracted from wbdc.names
data.columns = [
    "id",
    "diagnosis",
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",
    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave points_se",
    "symmetry_se",
    "fractal_dimension_se",
    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave points_worst",
    "symmetry_worst",
    "fractal_dimension_worst",
]

# save the data
data.to_csv("data.csv", sep=",", index=False)

data.sample(8)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
395,903811,B,14.06,17.18,89.75,609.1,0.08045,0.05361,0.02681,0.03251,...,14.92,25.34,96.42,684.5,0.1066,0.12310,0.08460,0.07911,0.2523,0.06609
257,886776,M,15.32,17.27,103.20,713.3,0.13350,0.22840,0.24480,0.12420,...,17.73,22.66,119.80,928.8,0.1765,0.45030,0.44290,0.22290,0.3258,0.11910
299,892399,B,10.51,23.09,66.85,334.2,0.10150,0.06797,0.02495,0.01875,...,10.93,24.22,70.10,362.7,0.1143,0.08614,0.04158,0.03125,0.2227,0.06777
452,9111843,B,12.00,28.23,76.77,442.5,0.08437,0.06450,0.04055,0.01945,...,13.09,37.88,85.07,523.7,0.1208,0.18560,0.18110,0.07116,0.2447,0.08194
375,901303,B,16.17,16.07,106.30,788.5,0.09880,0.14380,0.06651,0.05397,...,16.97,19.14,113.10,861.5,0.1235,0.25500,0.21140,0.12510,0.3153,0.08960
19,8510426,B,13.54,14.36,87.46,566.3,0.09779,0.08129,0.06664,0.04781,...,15.11,19.26,99.70,711.2,0.1440,0.17730,0.23900,0.12880,0.2977,0.07259
118,864877,M,15.78,22.91,105.70,782.6,0.11550,0.17520,0.21330,0.09479,...,20.19,30.50,130.30,1272.0,0.1855,0.49250,0.73560,0.20340,0.3274,0.12520
458,9112594,B,13.00,25.13,82.61,520.2,0.08369,0.05073,0.01206,0.01762,...,14.34,31.88,91.06,628.5,0.1218,0.10930,0.04462,0.05921,0.2306,0.06291


#### Key observations:
* The data has 569 observations and 32 columns.
* The first field is the 'id' attribute that we will want to drop before batch inference and add to the final inference output next to the probability of malignancy.
* Second field, 'diagnosis', is an indicator of the actual diagnosis ('M' = Malignant; 'B' = Benign).
* There are 30 other numeric features that we will use for training and inferencing.

Let's replace the M/B diagnosis with a 1/0 boolean value.

In [ ]:
data["diagnosis"] = data["diagnosis"].apply(lambda x: ((x == "M")) + 0)
data.sample(8)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
499,91485,1,20.590,21.24,137.80,1320.0,0.10850,0.16440,0.21880,0.11210,...,23.860,30.76,163.20,1760.0,0.14640,0.35970,0.51790,0.21130,0.2480,0.08999
301,892604,0,12.460,19.89,80.43,471.3,0.08451,0.10140,0.06830,0.03099,...,13.460,23.07,88.13,551.3,0.10500,0.21580,0.19040,0.07625,0.2685,0.07764
314,894047,0,8.597,18.60,54.09,221.2,0.10740,0.05847,0.00000,0.00000,...,8.952,22.44,56.65,240.1,0.13470,0.07767,0.00000,0.00000,0.3142,0.08116
506,91544001,0,12.220,20.04,79.47,453.1,0.10960,0.11520,0.08175,0.02166,...,13.160,24.17,85.13,515.3,0.14020,0.23150,0.35350,0.08088,0.2709,0.08839
450,9111596,0,11.870,21.54,76.83,432.0,0.06613,0.10640,0.08777,0.02386,...,12.790,28.18,83.51,507.2,0.09457,0.33990,0.32180,0.08750,0.2305,0.09952
244,884180,1,19.400,23.50,129.10,1155.0,0.10270,0.15580,0.20490,0.08886,...,21.650,30.53,144.90,1417.0,0.14630,0.29680,0.34580,0.15640,0.2920,0.07614
435,908489,1,13.980,19.62,91.12,599.5,0.10600,0.11330,0.11260,0.06463,...,17.040,30.80,113.90,869.3,0.16130,0.35680,0.40690,0.18270,0.3179,0.10550
520,917092,0,9.295,13.90,59.96,257.8,0.13710,0.12250,0.03332,0.02421,...,10.570,17.84,67.84,326.6,0.18500,0.20970,0.09996,0.07262,0.3681,0.08982


In [ ]:
# data split in three sets, training, validation and batch inference
rand_split = np.random.rand(len(data))
train_list = rand_split < 0.8
val_list = (rand_split >= 0.8) & (rand_split < 0.9)
batch_list = rand_split >= 0.9

data_train = data[train_list].drop(["id"], axis=1)
data_val = data[val_list].drop(["id"], axis=1)
data_batch = data[batch_list].drop(["diagnosis"], axis=1)
data_batch_noID = data_batch.drop(["id"], axis=1)

Let's upload those data sets in S3

In [ ]:
train_file = "train_data.csv"
data_train.to_csv(train_file, index=False, header=False)
sess.upload_data(train_file, key_prefix="{}/train".format(prefix))

validation_file = "validation_data.csv"
data_val.to_csv(validation_file, index=False, header=False)
sess.upload_data(validation_file, key_prefix="{}/validation".format(prefix))

batch_file = "batch_data.csv"
data_batch.to_csv(batch_file, index=False, header=False)
sess.upload_data(batch_file, key_prefix="{}/batch".format(prefix))

batch_file_noID = "batch_data_noID.csv"
data_batch_noID.to_csv(batch_file_noID, index=False, header=False)
sess.upload_data(batch_file_noID, key_prefix="{}/batch".format(prefix))

's3://sagemaker-us-east-1-239153173561/DEMO-breast-cancer-prediction-xgboost-highlevel/batch/batch_data_noID.csv'

---

## Training job and model creation

In [ ]:
%%time
from time import gmtime, strftime

job_name = "xgb-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
output_location = "s3://{}/{}/output/{}".format(bucket, prefix, job_name)
image = sagemaker.image_uris.retrieve(
    framework="xgboost", region=boto3.Session().region_name, version="1.7-1"
)

sm_estimator = sagemaker.estimator.Estimator(
    image,
    role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size=50,
    input_mode="File",
    output_path=output_location,
    sagemaker_session=sess,
)

sm_estimator.set_hyperparameters(
    objective="binary:logistic",
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    verbosity=0,
    num_round=100,
)

train_data = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train".format(bucket, prefix),
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
validation_data = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validation".format(bucket, prefix),
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
data_channels = {"train": train_data, "validation": validation_data}

# Start training by calling the fit method in the estimator
sm_estimator.fit(inputs=data_channels, job_name=job_name, logs=True)

INFO:sagemaker:Creating training-job with name: xgb-2026-05-30-05-23-35


2026-05-30 05:23:37 Starting - Starting the training job.

.

.


2026-05-30 05:23:53 Starting - Preparing the instances for training.

...

.


2026-05-30 05:25:32 Training - Training image download completed. Training in progress..

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30 05:25:43.043 ip-10-0-78-168.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-05-30 05:25:43.119 ip-10-0-78-168.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-05-30:05:25:43:INFO] Imported framework sagemaker_xgboost_container.training
[2026-05-30:05:25:43:INFO] Failed to parse hyperparameter objective value binary:logistic to Json.
Returning the value itself
[2026-05-30:05:25:43:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:25:43:INFO] Running XGBoost Sagemaker in algorithm mode
[2026-05-30:05:25:43:INFO] Determined 0 GPU(s) available on the instance.
[20


2026-05-30 05:26:01 Uploading - Uploading generated training model
2026-05-30 05:26:01 Completed - Training job completed


Training seconds: 104
Billable seconds: 104
CPU times: user 574 ms, sys: 23 ms, total: 597 ms
Wall time: 2min 47s


---

## Batch Transform

### 1. Create a transform job with ther default configurations




In [ ]:
%%time

sm_transformer = sm_estimator.transformer(1, "ml.m5.xlarge")

# start a transform job
input_location = "s3://{}/{}/batch/{}".format(
    bucket, prefix, batch_file_noID
)  # use input data without ID column
sm_transformer.transform(input_location, content_type="text/csv", split_type="Line")
sm_transformer.wait()


INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-05-30-05-26-33-850


INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-05-30-05-26-34-539


.

.

.

...

.

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30:05:31:53:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:31:53:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:31:53:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) 

2026-05-30T05:31:59.060:[sagemaker logs]: MaxConcurrentTransforms=4, MaxPayloadInMB=6, BatchStrategy=MULTI_RECORD


/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30:05:31:53:INFO] No GPUs detected (normal if no gpus installed)
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30:05:31:53:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:31:53:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:31:53:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;

Let's inspect the output of the Batch Transform job in S3. It should show the list probabilities of tumors being malignant.

In [ ]:
import re


def get_csv_output_from_s3(s3uri, batch_file):
    file_name = "{}.out".format(batch_file)
    match = re.match("s3://([^/]+)/(.*)", "{}/{}".format(s3uri, file_name))
    output_bucket, output_prefix = match.group(1), match.group(2)
    s3.download_file(output_bucket, output_prefix, file_name)
    return pd.read_csv(file_name, sep=",", header=None)

In [ ]:
output_df = get_csv_output_from_s3(sm_transformer.output_path, batch_file_noID)
output_df.head(8)

,0
0,0.992567
1,0.880336
2,0.885230
3,0.009095
4,0.006452
5,0.014160
6,0.977159
7,0.891755


#### 2. Join the input and the prediction results


In [ ]:
# content_type / accept and split_type / assemble_with are required to use IO joining feature
sm_transformer.assemble_with = "Line"
sm_transformer.accept = "text/csv"

# start a transform job
input_location = "s3://{}/{}/batch/{}".format(
    bucket, prefix, batch_file
)  # use input data with ID column cause InputFilter will filter it out
sm_transformer.transform(
    input_location,
    split_type="Line",
    content_type="text/csv",
    input_filter="$[1:]",
    join_source="Input",
)
sm_transformer.wait()

INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-05-30-05-33-33-499


.

.

.

.

...

.

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30:05:38:42:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:38:42:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:38:42:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) 

2026-05-30T05:38:48.142:[sagemaker logs]: MaxConcurrentTransforms=4, MaxPayloadInMB=6, BatchStrategy=MULTI_RECORD


/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30:05:38:42:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:38:42:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:38:42:INFO] nginx config: 
worker_processes auto;
daemon off;
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30:05:38:42:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:38:4

Let's inspect the output of the Batch Transform job in S3. It should show the list of tumors identified by their original feature columns and their corresponding probabilities of being malignant.

In [ ]:
output_df = get_csv_output_from_s3(sm_transformer.output_path, batch_file)
output_df.head(8)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,84300903,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.197400,0.127900,0.2069,...,25.53,152.50,1709.0,0.1444,0.42450,0.450400,0.24300,0.3613,0.08758,0.992567
1,853201,17.57,15.05,115.00,955.1,0.09847,0.11570,0.098750,0.079530,0.1739,...,19.52,134.90,1227.0,0.1255,0.28120,0.248900,0.14560,0.2756,0.07919,0.880336
2,857392,18.22,18.70,120.30,1033.0,0.11480,0.14850,0.177200,0.106000,0.2092,...,24.13,135.10,1321.0,0.1280,0.22970,0.262300,0.13250,0.3021,0.07987,0.885230
3,857810,13.05,19.31,82.61,527.2,0.08060,0.03789,0.000692,0.004167,0.1819,...,22.25,90.24,624.1,0.1021,0.06191,0.001845,0.01111,0.2439,0.06289,0.009095
4,859487,12.78,16.49,81.37,502.5,0.09831,0.05234,0.036530,0.028640,0.1590,...,19.76,85.67,554.9,0.1296,0.07061,0.103900,0.05882,0.2383,0.06410,0.006452
5,865137,11.41,10.82,73.34,403.3,0.09373,0.06685,0.035120,0.026230,0.1667,...,15.97,83.74,510.5,0.1548,0.23900,0.210200,0.08958,0.3016,0.08523,0.014160
6,86730502,16.16,21.54,106.20,809.8,0.10080,0.12840,0.104300,0.056130,0.2160,...,31.68,129.70,1175.0,0.1395,0.30550,0.299200,0.13120,0.3480,0.07619,0.977159
7,869104,16.11,18.05,105.10,813.0,0.09721,0.11370,0.094470,0.059430,0.1861,...,25.27,129.00,1233.0,0.1314,0.22360,0.280200,0.12160,0.2792,0.08158,0.891755


#### 3. Update the output filter to keep only ID and prediction results

In [ ]:
# start another transform job
sm_transformer.transform(
    input_location,
    split_type="Line",
    content_type="text/csv",
    input_filter="$[1:]",
    join_source="Input",
    output_filter="$[0,-1]",
)
sm_transformer.wait()

INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-05-30-05-39-52-616


.

.

.

.

...

.

.

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30:05:45:09:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:45:09:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:45:09:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  impor

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30:05:45:09:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:45:09:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-30:05:45:09:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  impor

Now, let's inspect the output of the Batch Transform job in S3 again. It should show 2 columns: the ID and their corresponding probabilities of being malignant.

In [ ]:
output_df = get_csv_output_from_s3(sm_transformer.output_path, batch_file)
output_df.head(8)

,0,1
0,84300903,0.992567
1,853201,0.880336
2,857392,0.885230
3,857810,0.009095
4,859487,0.006452
5,865137,0.014160
6,86730502,0.977159
7,869104,0.891755


## Upload the Sagemaker Model created during our training job to the Sagemaker Model Registry

In [ ]:
sagemaker = boto3.client("sagemaker")

model_name = job_name
print(model_name)


info = sagemaker.describe_training_job(TrainingJobName=model_name)
model_data = info["ModelArtifacts"]["S3ModelArtifacts"]

primary_container = {"Image": image, "ModelDataUrl": model_data}

# Save our model to the Sagemaker Model Registry
create_model_response = sagemaker.create_model(
    ModelName=model_name, ExecutionRoleArn=role, PrimaryContainer=primary_container
)

print(create_model_response["ModelArn"])

xgb-2026-05-30-05-23-35


arn:aws:sagemaker:us-east-1:239153173561:model/xgb-2026-05-30-05-23-35


In [ ]:
# Inspect Training Job Details
info

{'TrainingJobName': 'xgb-2026-05-30-05-23-35',
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:239153173561:training-job/xgb-2026-05-30-05-23-35',
 'ModelArtifacts': {'S3ModelArtifacts': 's3://sagemaker-us-east-1-239153173561/DEMO-breast-cancer-prediction-xgboost-highlevel/output/xgb-2026-05-30-05-23-35/xgb-2026-05-30-05-23-35/output/model.tar.gz'},
 'TrainingJobStatus': 'Completed',
 'SecondaryStatus': 'Completed',
 'HyperParameters': {'eta': '0.2',
  'gamma': '4',
  'max_depth': '5',
  'min_child_weight': '6',
  'num_round': '100',
  'objective': 'binary:logistic',
  'subsample': '0.8',
  'verbosity': '0'},
 'AlgorithmSpecification': {'TrainingImage': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1',
  'TrainingInputMode': 'File',
  'MetricDefinitions': [{'Name': 'train:mae',
    'Regex': '.*\\[[0-9]+\\].*#011train-mae:([-+]?[0-9]*\\.?[0-9]+(?:[eE][-+]?[0-9]+)?).*'},
   {'Name': 'validation:aucpr',
    'Regex': '.*\\[[0-9]+\\].*#011validation-aucpr:([-+]?[0-9]*\

In [ ]:
# Create Endpoint Configuration


# Create an endpoint config name. Here we create one based on the date
# so it we can search endpoints based on creation time.
endpoint_config_name = 'lab4-1-endpoint-config' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())

instance_type = 'ml.m5.xlarge'

endpoint_config_response = sagemaker.create_endpoint_config(
    EndpointConfigName=endpoint_config_name, # You will specify this name in a CreateEndpoint request.
    # List of ProductionVariant objects, one for each model that you want to host at this endpoint.
    ProductionVariants=[
        {
            "VariantName": "variant1", # The name of the production variant.
            "ModelName": model_name,
            "InstanceType": instance_type, # Specify the compute instance type.
            "InitialInstanceCount": 1 # Number of instances to launch initially.
        }
    ]
)

print(f"Created EndpointConfig: {endpoint_config_response['EndpointConfigArn']}")


Created EndpointConfig: arn:aws:sagemaker:us-east-1:239153173561:endpoint-config/lab4-1-endpoint-config2026-05-30-05-46-36


In [ ]:
# Deploy our model to real-time endpoint

endpoint_name = 'lab4-1-endpoint' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())


create_endpoint_response = sagemaker.create_endpoint(
                                            EndpointName=endpoint_name,
                                            EndpointConfigName=endpoint_config_name)

In [ ]:
# Wait for endpoint to spin up
from time import sleep

sagemaker.describe_endpoint(EndpointName=endpoint_name)

while True:
    print("Getting Job Status")
    res = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    state = res["EndpointStatus"]

    if state == "InService":
        print("Endpoint in Service")
        break
    elif state == "Creating":
        print("Endpoint still creating...")
        sleep(60)
    else:
        print("Endpoint Creation Error - Check Sagemaker Console")
        break

Getting Job Status
Endpoint still creating...


Getting Job Status
Endpoint still creating...


Getting Job Status
Endpoint still creating...


Getting Job Status
Endpoint in Service


In [ ]:
# Invoke Endpoint

sagemaker_runtime = boto3.client("sagemaker-runtime", region_name=region)

response = sagemaker_runtime.invoke_endpoint(
                            EndpointName=endpoint_name,
                            ContentType='text/csv',
                            Body=data_batch_noID.to_csv(header=None, index=False).strip('\n').split('\n')[0]
                            )
print(response['Body'].read().decode('utf-8'))

0.9925665855407715



In [ ]:
# Examine Response Body

response

{'ResponseMetadata': {'RequestId': '611c890c-e5c2-4606-a9c4-715d92d26c9d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '611c890c-e5c2-4606-a9c4-715d92d26c9d',
   'x-amzn-invoked-production-variant': 'variant1',
   'date': 'Sat, 30 May 2026 05:51:45 GMT',
   'content-type': 'text/csv; charset=utf-8',
   'content-length': '19',
   'connection': 'keep-alive'},
  'RetryAttempts': 0},
 'ContentType': 'text/csv; charset=utf-8',
 'InvokedProductionVariant': 'variant1',
 'Body': <botocore.response.StreamingBody at 0x7f9f97f1f6a0>}

# **Part 1: Set Up Model Group**

Model Group will track versions of the XGBoost breast cancer classification model.

In [ ]:
import time

# Informative name for the Model Group
# The timestamp prevents an error if a group with the same base name already exists
model_package_group_name = "xgboost-breast-cancer-detection-" + str(round(time.time()))

# Brief description of the purpose of this Model Group
model_package_group_description = (
    "Model group for XGBoost models that classify breast tumors "
    "as malignant or benign using diagnostic measurements."
)

# Create the Model Group
create_model_package_group_response = sagemaker.create_model_package_group(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageGroupDescription=model_package_group_description
)

# Display the Model Group ARN
print("Model Package Group Name:", model_package_group_name)
print("Model Package Group ARN:", create_model_package_group_response["ModelPackageGroupArn"])

Model Package Group Name: xgboost-breast-cancer-detection-1780120348
Model Package Group ARN: arn:aws:sagemaker:us-east-1:239153173561:model-package-group/xgboost-breast-cancer-detection-1780120348


In [ ]:
# Describe the Model Group

describe_model_package_group_response = sagemaker.describe_model_package_group(
    ModelPackageGroupName=model_package_group_name
)

describe_model_package_group_response

{'ModelPackageGroupName': 'xgboost-breast-cancer-detection-1780120348',
 'ModelPackageGroupArn': 'arn:aws:sagemaker:us-east-1:239153173561:model-package-group/xgboost-breast-cancer-detection-1780120348',
 'ModelPackageGroupDescription': 'Model group for XGBoost models that classify breast tumors as malignant or benign using diagnostic measurements.',
 'CreationTime': datetime.datetime(2026, 5, 30, 5, 52, 28, 578000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:239153173561:user-profile/d-hzdybklriydx/default-1778958208170',
  'UserProfileName': 'default-1778958208170',
  'DomainId': 'd-hzdybklriydx',
  'IamIdentity': {'Arn': 'arn:aws:sts::239153173561:assumed-role/LabRole/SageMaker',
   'PrincipalId': 'AROATPLVD2Q46XBOXWE26:SageMaker'}},
 'ModelPackageGroupStatus': 'Completed',
 'ResponseMetadata': {'RequestId': '971510d9-f5b6-4bfc-bcfe-e1e371bc3610',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '971510d9-f5b6-4bfc-bcfe-e1e371bc3610

# **Part 2: Set Up Model Package**

Model Package registers the trained XGBoost breast cancer classification model as a version within the Model Group. It documents the model artifact location, inference container, supported CSV input/output format, and approval status.

In [ ]:
# Create a Model Package for the trained XGBoost model

# Define the model deployment information for the registered model version
modelpackage_inference_specification = {
    "InferenceSpecification": {
        "Containers": [
            {
                "Image": image,
                "ModelDataUrl": model_data
            }
        ],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"]
    }
}

# Provide registration information for this specific model version
create_model_package_input_dict = {
    "ModelPackageGroupName": model_package_group_name,
    "ModelPackageDescription": (
        "XGBoost breast cancer classification model that predicts whether "
        "a tumor is malignant or benign."
    ),
    "ModelApprovalStatus": "PendingManualApproval"
}

# Combine the registration details with the inference information
create_model_package_input_dict.update(modelpackage_inference_specification)

# Register this model version inside the Model Group
create_model_package_response = sagemaker.create_model_package(
    **create_model_package_input_dict
)

# Save and display the ARN of the new Model Package
model_package_arn = create_model_package_response["ModelPackageArn"]

print("Model Package Version ARN:")
print(model_package_arn)

Model Package Version ARN:
arn:aws:sagemaker:us-east-1:239153173561:model-package/xgboost-breast-cancer-detection-1780120348/1


In [ ]:
# Describe the Model Package

describe_model_package_response = sagemaker.describe_model_package(
    ModelPackageName=model_package_arn
)

describe_model_package_response

{'ModelPackageGroupName': 'xgboost-breast-cancer-detection-1780120348',
 'ModelPackageVersion': 1,
 'ModelPackageRegistrationType': 'Registered',
 'ModelPackageArn': 'arn:aws:sagemaker:us-east-1:239153173561:model-package/xgboost-breast-cancer-detection-1780120348/1',
 'ModelPackageDescription': 'XGBoost breast cancer classification model that predicts whether a tumor is malignant or benign.',
 'CreationTime': datetime.datetime(2026, 5, 30, 6, 0, 43, 807000, tzinfo=tzlocal()),
 'InferenceSpecification': {'Containers': [{'Image': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1',
    'ImageDigest': 'sha256:b4f13edb198529c460692015797fa1ca6a8ff1ed64a149297174d922121b8fc4',
    'ModelDataUrl': 's3://sagemaker-us-east-1-239153173561/DEMO-breast-cancer-prediction-xgboost-highlevel/output/xgb-2026-05-30-05-23-35/xgb-2026-05-30-05-23-35/output/model.tar.gz',
    'ModelDataETag': '2c831b4e3c8a1dc92a86499cba7defa7',
    'IsCheckpoint': False}],
  'SupportedContentTypes': ['

# **Part 3: Write the Model Card**

Model Card documents the XGBoost breast cancer classification model registered in the SageMaker Model Registry. It records the model purpose, intended use, training information, input features, deployment testing, limitations, and ethical considerations for future review and maintenance.

In [ ]:
# Creating a Model Card for the registered XGBoost model

import json
import time

# Create a unique name for the Model Card
model_card_name = "xgboost-breast-cancer-model-card-" + str(round(time.time()))

# Define the Model Card content
model_card_content = {
    "model_overview": {
        "model_description": (
            "XGBoost binary classification model used to predict whether "
            "a breast tumor is malignant or benign."
        ),
        "model_creator": "Jose Sandoval",
        "model_owner": "Jose Sandoval",
        "model_artifact": [model_data],
        "algorithm_type": "XGBoost",
        "problem_type": "Binary Classification"
    },

    "intended_uses": {
        "purpose_of_model": (
            "Predict whether a breast tumor is malignant or benign using "
            "numeric diagnostic measurement features."
        ),
        "intended_uses": (
            "This model is intended for coursework demonstration of model training, "
            "batch transform, real-time endpoint deployment, and SageMaker Model Registry."
        ),
        "factors_affecting_model_efficiency": (
            "Model performance may be affected by missing measurements, changes in "
            "patient populations, data quality issues, or feature distributions that "
            "differ from the training dataset."
        ),
        "risk_rating": "Medium",
        "explanations_for_risk_rating": (
            "Although this model is used for an educational assignment, breast cancer "
            "classification relates to healthcare decisions. The model should not be "
            "used as a medical diagnosis or replace clinical judgment."
        )
    },

    "business_details": {
        "business_problem": (
            "Classify breast tumor observations as malignant or benign using "
            "diagnostic measurement data."
        ),
        "business_stakeholders": (
            "Model developer, course evaluator, and future model maintainers."
        ),
        "line_of_business": "Educational healthcare machine learning demonstration"
    },

    "training_details": {
        "objective_function": {
            "function": {
                "function": "Minimize",
                "facet": "Loss",
                "condition": "Binary classification loss"
            },
            "notes": (
                "The binary:logistic objective was used to generate probabilities "
                "for malignant versus benign tumor classification."
            )
        },
        "training_observations": (
            "The model was trained in Amazon SageMaker using the XGBoost 1.7-1 "
            "container on breast cancer diagnostic measurement data. Input features "
            "include numeric tumor measurements such as radius, texture, perimeter, "
            "area, smoothness, compactness, concavity, symmetry, and fractal dimension."
        ),
        "training_job_details": {
            "user_provided_hyper_parameters": [
                {"name": "objective", "value": "binary:logistic"},
                {"name": "max_depth", "value": "5"},
                {"name": "eta", "value": "0.2"},
                {"name": "gamma", "value": "4"},
                {"name": "min_child_weight", "value": "6"},
                {"name": "subsample", "value": "0.8"},
                {"name": "verbosity", "value": "0"},
                {"name": "num_round", "value": "100"}
            ]
        }
    },

    "evaluation_details": [
        {
            "name": "deployment-validation",
            "evaluation_observation": (
                "The trained model was successfully tested through a SageMaker "
                "batch transform job and a real-time endpoint inference request."
            )
        }
    ],

    "additional_information": {
        "ethical_considerations": (
            "This model should not be used independently for medical diagnosis or "
            "treatment decisions. Real-world use would require clinical validation, "
            "privacy protections, monitoring, and bias evaluation."
        ),
        "caveats_and_recommendations": (
            "This model was developed for an educational assignment. Future versions "
            "should document evaluation metrics, monitoring reports, and any changes "
            "to training data or hyperparameters."
        ),
        "custom_details": {
            "Model Package ARN": model_package_arn,
            "Model Package Group": model_package_group_name,
            "Model Version": "1",
            "Input Format": "text/csv",
            "Output Format": "text/csv"
        }
    }
}

# Create the Model Card
create_model_card_response = sagemaker.create_model_card(
    ModelCardName=model_card_name,
    Content=json.dumps(model_card_content),
    ModelCardStatus="Draft"
)

# Display the Model Card information
model_card_arn = create_model_card_response["ModelCardArn"]

print("Model Card Name:", model_card_name)
print("Model Card ARN:", model_card_arn)

Model Card Name: xgboost-breast-cancer-model-card-1780121778
Model Card ARN: arn:aws:sagemaker:us-east-1:239153173561:model-card/xgboost-breast-cancer-model-card-1780121778


In [ ]:
# Describe the Model Card

describe_model_card_response = sagemaker.describe_model_card(
    ModelCardName=model_card_name
)

describe_model_card_response

{'ModelCardArn': 'arn:aws:sagemaker:us-east-1:239153173561:model-card/xgboost-breast-cancer-model-card-1780121778',
 'ModelCardName': 'xgboost-breast-cancer-model-card-1780121778',
 'ModelCardVersion': 1,
 'Content': '{"model_overview": {"model_description": "XGBoost binary classification model used to predict whether a breast tumor is malignant or benign.", "model_creator": "Jose Sandoval", "model_owner": "Jose Sandoval", "model_artifact": ["s3://sagemaker-us-east-1-239153173561/DEMO-breast-cancer-prediction-xgboost-highlevel/output/xgb-2026-05-30-05-23-35/xgb-2026-05-30-05-23-35/output/model.tar.gz"], "algorithm_type": "XGBoost", "problem_type": "Binary Classification"}, "intended_uses": {"purpose_of_model": "Predict whether a breast tumor is malignant or benign using numeric diagnostic measurement features.", "intended_uses": "This model is intended for coursework demonstration of model training, batch transform, real-time endpoint deployment, and SageMaker Model Registry.", "factor

In [ ]:
# Delete Endpoint

sagemaker.delete_endpoint(EndpointName=endpoint_name)

{'ResponseMetadata': {'RequestId': '92aae405-5bf6-4dc0-83ba-9db7162a65a0',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '92aae405-5bf6-4dc0-83ba-9db7162a65a0',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Sat, 30 May 2026 06:20:28 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}